# LeetCode 1652 — Defuse the Bomb bằng Python

Link đề: https://leetcode.com/problems/defuse-the-bomb/

Độ khó: **Easy** · Chủ đề: Array, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def decrypt(self, code: list[int], k: int) -> list[int]:

```

Mục tiêu học:

1. Hiểu **mảng vòng** (circular array) và dùng phép `% n` để "quay vòng" chỉ số.
2. Viết được cách trực giác `O(n · |k|)` cộng từng phần tử.
3. Nâng lên **cửa sổ trượt** (sliding window) `O(n)`: thêm phần tử mới vào, bỏ phần tử cũ ra.
4. Biết thêm hai kỹ thuật: **nhân đôi mảng** và **tổng tiền tố** (prefix sum).
5. Xử lý đúng ba trường hợp `k > 0`, `k < 0`, `k == 0`, và hiểu "thay thế đồng thời".
6. Tự viết lại lời giải và giải các biến thể cửa sổ trên mảng vòng.

## 1. Bài toán yêu cầu gì?

Cho mảng vòng `code` dài `n` và khóa `k`. Thay **mọi** phần tử **cùng lúc** (đồng thời):

- `k > 0`: phần tử thứ `i` thành **tổng `k` phần tử tiếp theo** (bên phải).
- `k < 0`: phần tử thứ `i` thành **tổng `-k` phần tử phía trước** (bên trái).
- `k == 0`: phần tử thứ `i` thành `0`.

"Mảng vòng": sau `code[n-1]` là `code[0]`; trước `code[0]` là `code[n-1]`.

| code | k | Kết quả | Giải thích |
|---|---:|---|---|
| `[5,7,1,4]` | `3` | `[12,10,16,13]` | `[7+1+4, 1+4+5, 4+5+7, 5+7+1]` |
| `[1,2,3,4]` | `0` | `[0,0,0,0]` | `k = 0` |
| `[2,4,9,3]` | `-2` | `[12,5,6,13]` | `[3+9, 2+3, 4+2, 9+4]` |

Constraints:

- `n == code.length`, `1 <= n <= 100`
- `1 <= code[i] <= 100`
- `-(n - 1) <= k <= n - 1` — nên cửa sổ không bao giờ chứa chính phần tử `i`.

**"Đồng thời"** nghĩa là mọi tổng đều tính trên mảng **gốc**. Ta phải ghi kết quả ra mảng **mới**, không sửa `code` khi đang tính.

### Tính tay ví dụ 1 (`k = 3`)

```text
chỉ số:   0  1  2  3 | 0  1  2   (phần lặp lại để thấy vòng)
code:     5  7  1  4 | 5  7  1

i = 0: lấy chỉ số 1,2,3 -> 7 + 1 + 4 = 12
i = 1: lấy chỉ số 2,3,0 -> 1 + 4 + 5 = 10
i = 2: lấy chỉ số 3,0,1 -> 4 + 5 + 7 = 16
i = 3: lấy chỉ số 0,1,2 -> 5 + 7 + 1 = 13
```

Ví dụ 3 (`k = -2`): `i = 0` lấy hai phần tử trước là chỉ số `3, 2` → `3 + 9 = 12`.

In [1]:
code = [5, 7, 1, 4]
n = len(code)
for i in range(n):
    idx = [(i + j) % n for j in range(1, 4)]
    print(f"i={i}: chỉ số {idx} -> {[code[t] for t in idx]} tổng {sum(code[t] for t in idx)}")

i=0: chỉ số [1, 2, 3] -> [7, 1, 4] tổng 12
i=1: chỉ số [2, 3, 0] -> [1, 4, 5] tổng 10
i=2: chỉ số [3, 0, 1] -> [4, 5, 7] tổng 16
i=3: chỉ số [0, 1, 2] -> [5, 7, 1] tổng 13


## 2. Nền tảng Python cần biết

### 2.1. Phép `%` để quay vòng chỉ số

`(i + j) % n` luôn nằm trong `0..n-1`. Khi `i + j` vượt `n - 1`, nó "quay về đầu".

In [2]:
n = 4
for x in range(0, 9):
    print(f"{x} % {n} = {x % n}")

0 % 4 = 0
1 % 4 = 1
2 % 4 = 2
3 % 4 = 3
4 % 4 = 0
5 % 4 = 1
6 % 4 = 2
7 % 4 = 3
8 % 4 = 0


### 2.2. `%` với số âm trong Python

Python luôn cho `a % n` **cùng dấu với `n`** (với `n > 0` là kết quả không âm). Vì vậy `(-1) % 4 == 3` — đúng là "phần tử trước chỉ số 0 là chỉ số 3".

Chú ý: C++/Java cho `-1 % 4 == -1`, phải tự cộng thêm `n`. Python không cần.

In [3]:
n = 4
for x in [-1, -2, -3, -4, -5]:
    print(f"{x} % {n} = {x % n}")

-1 % 4 = 3
-2 % 4 = 2
-3 % 4 = 1
-4 % 4 = 0
-5 % 4 = 3


### 2.3. Đọc chỉ số âm trực tiếp

Python cho phép `code[-1]` là phần tử cuối, `code[-2]` là áp cuối. Nhưng `code[-5]` với list dài 4 sẽ lỗi. Dùng `% n` an toàn hơn.

In [4]:
code = [2, 4, 9, 3]
print(code[-1], code[-2])
try:
    print(code[-5])
except IndexError as e:
    print("IndexError:", e)
print(code[-5 % 4])

3 9
IndexError: list index out of range
3


### 2.4. Tạo mảng kết quả mới: `[0] * n`

Vì phải thay "đồng thời", ta tạo `res = [0] * n` và ghi vào đó, giữ `code` nguyên.

In [5]:
code = [5, 7, 1, 4]
res = [0] * len(code)
res[0] = 12
print("code vẫn giữ nguyên:", code)
print("res:", res)

code vẫn giữ nguyên: [5, 7, 1, 4]
res: [12, 0, 0, 0]


### 2.5. `sum()` trên generator

`sum(code[(i + j) % n] for j in range(1, k + 1))` cộng `k` phần tử tiếp theo của `i`.

In [6]:
code, n, i, k = [5, 7, 1, 4], 4, 2, 3
print(sum(code[(i + j) % n] for j in range(1, k + 1)))

16


## 3. Cách trực giác — cộng từng phần tử với `% n`

Với mỗi `i`:

- `k > 0`: cộng `code[(i + j) % n]` với `j = 1..k`.
- `k < 0`: cộng `code[(i - j) % n]` với `j = 1..-k`.
- `k == 0`: `0`.

In [7]:
def decrypt_bruteforce(code: list[int], k: int) -> list[int]:
    n = len(code)
    res = [0] * n
    if k == 0:
        return res
    for i in range(n):
        total = 0
        if k > 0:
            for j in range(1, k + 1):
                total += code[(i + j) % n]
        else:
            for j in range(1, -k + 1):
                total += code[(i - j) % n]
        res[i] = total
    return res

print(decrypt_bruteforce([5, 7, 1, 4], 3))
print(decrypt_bruteforce([1, 2, 3, 4], 0))
print(decrypt_bruteforce([2, 4, 9, 3], -2))

[12, 10, 16, 13]
[0, 0, 0, 0]
[12, 5, 6, 13]


### Nút thắt

Mỗi `i` cộng `|k|` số → tổng `O(n · |k|)`, xấu nhất `O(n²)` khi `|k| = n - 1`.

Để ý hai cửa sổ liền nhau **chồng lên nhau gần hết**:

```text
i = 0 (k=3): chỉ số {1, 2, 3}
i = 1 (k=3): chỉ số {2, 3, 0}
```

Chung `{2, 3}`! Từ tổng của `i = 0`, chỉ cần **bỏ `code[1]`, thêm `code[0]`** là có tổng của `i = 1`. Ta đang cộng lại những thứ đã cộng.

In [8]:
code = [5, 7, 1, 4]
s0 = 7 + 1 + 4
s1 = s0 - code[1] + code[(1 + 3) % 4]
print("tổng i=0:", s0)
print("tổng i=1 = s0 - code[1] + code[0] =", s1)

tổng i=0: 12
tổng i=1 = s0 - code[1] + code[0] = 10


## 4. Insight — cửa sổ trượt có độ dài |k|

Cả hai trường hợp `k > 0` và `k < 0` đều là một **cửa sổ liên tiếp dài `|k|`** trên mảng vòng:

- `k > 0`: cửa sổ của `i` là `[i + 1, i + k]`.
- `k < 0`: cửa sổ của `i` là `[i + k, i - 1]` (vì `k` âm, `i + k < i`).

Gọi `start`, `end` là hai đầu cửa sổ của `i = 0`:

| trường hợp | start | end |
|---|---:|---:|
| `k > 0` | `1` | `k` |
| `k < 0` | `n + k` | `n - 1` |

Khi `i` tăng 1, **cả hai đầu cùng dịch sang phải 1** (theo modulo `n`):

```text
window_sum = window_sum - code[start % n] + code[(end + 1) % n]
start += 1; end += 1
```

In [9]:
n, k = 4, -2
start, end = n + k, n - 1
print(f"k={k}: cửa sổ của i=0 là chỉ số {start}..{end}")
for i in range(n):
    print(f"i={i}: cửa sổ {[t % n for t in range(start + i, end + i + 1)]}")

k=-2: cửa sổ của i=0 là chỉ số 2..3
i=0: cửa sổ [2, 3]
i=1: cửa sổ [3, 0]
i=2: cửa sổ [0, 1]
i=3: cửa sổ [1, 2]


## 5. Lời giải chính — sliding window `O(n)`

Giảng từng đoạn:

**Đoạn 1 — `k == 0`:** trả `[0] * n`.

**Đoạn 2 — xác định cửa sổ đầu tiên** (`start`, `end`) theo dấu của `k`.

**Đoạn 3 — tính tổng cửa sổ đầu** bằng một vòng lặp ngắn.

**Đoạn 4 — trượt:** với mỗi `i`, ghi `res[i] = window_sum`, rồi bỏ `code[start]`, thêm `code[end + 1]`, dịch hai đầu.

In [10]:
def decrypt_window(code: list[int], k: int) -> list[int]:
    n = len(code)
    res = [0] * n
    if k == 0:
        return res
    if k > 0:
        start, end = 1, k
    else:
        start, end = n + k, n - 1
    window_sum = sum(code[t % n] for t in range(start, end + 1))
    for i in range(n):
        res[i] = window_sum
        window_sum -= code[start % n]
        window_sum += code[(end + 1) % n]
        start += 1
        end += 1
    return res

print(decrypt_window([5, 7, 1, 4], 3))
print(decrypt_window([1, 2, 3, 4], 0))
print(decrypt_window([2, 4, 9, 3], -2))

[12, 10, 16, 13]
[0, 0, 0, 0]
[12, 5, 6, 13]


## 6. TRACE từng bước

Trace `code = [2,4,9,3]`, `k = -2`: cửa sổ đầu `start = 2`, `end = 3`, tổng `9 + 3 = 12`.

| i | cửa sổ (chỉ số) | window_sum ghi vào res[i] | bỏ ra | thêm vào | tổng mới |
|---:|---|---:|---|---|---:|
| 0 | 2,3 | 12 | code[2]=9 | code[0]=2 | 5 |
| 1 | 3,0 | 5 | code[3]=3 | code[1]=4 | 6 |
| 2 | 0,1 | 6 | code[0]=2 | code[2]=9 | 13 |
| 3 | 1,2 | 13 | code[1]=4 | code[3]=3 | 12 |

Kết quả `[12, 5, 6, 13]` đúng như đề.

In [11]:
def trace_decrypt(code: list[int], k: int) -> list[int]:
    n = len(code)
    res = [0] * n
    if k == 0:
        print("k = 0 -> toàn 0")
        return res
    start, end = (1, k) if k > 0 else (n + k, n - 1)
    window_sum = sum(code[t % n] for t in range(start, end + 1))
    print(f"code={code}, k={k}, cửa sổ đầu {start}..{end}, tổng={window_sum}")
    for i in range(n):
        win = [t % n for t in range(start, end + 1)]
        res[i] = window_sum
        out_i, in_i = start % n, (end + 1) % n
        new_sum = window_sum - code[out_i] + code[in_i]
        print(f"i={i}: cửa sổ {win} -> res[{i}]={window_sum} | bỏ code[{out_i}]={code[out_i]}, "
              f"thêm code[{in_i}]={code[in_i]} -> {new_sum}")
        window_sum = new_sum
        start += 1
        end += 1
    print("Kết quả:", res)
    return res

trace_decrypt([2, 4, 9, 3], -2)

code=[2, 4, 9, 3], k=-2, cửa sổ đầu 2..3, tổng=12
i=0: cửa sổ [2, 3] -> res[0]=12 | bỏ code[2]=9, thêm code[0]=2 -> 5
i=1: cửa sổ [3, 0] -> res[1]=5 | bỏ code[3]=3, thêm code[1]=4 -> 6
i=2: cửa sổ [0, 1] -> res[2]=6 | bỏ code[0]=2, thêm code[2]=9 -> 13
i=3: cửa sổ [1, 2] -> res[3]=13 | bỏ code[1]=4, thêm code[3]=3 -> 12
Kết quả: [12, 5, 6, 13]


[12, 5, 6, 13]

In [12]:
trace_decrypt([5, 7, 1, 4], 3)

code=[5, 7, 1, 4], k=3, cửa sổ đầu 1..3, tổng=12
i=0: cửa sổ [1, 2, 3] -> res[0]=12 | bỏ code[1]=7, thêm code[0]=5 -> 10
i=1: cửa sổ [2, 3, 0] -> res[1]=10 | bỏ code[2]=1, thêm code[1]=7 -> 16
i=2: cửa sổ [3, 0, 1] -> res[2]=16 | bỏ code[3]=4, thêm code[2]=1 -> 13
i=3: cửa sổ [0, 1, 2] -> res[3]=13 | bỏ code[0]=5, thêm code[3]=4 -> 12
Kết quả: [12, 10, 16, 13]


[12, 10, 16, 13]

Dòng `start, end = (1, k) if k > 0 else (n + k, n - 1)` là **biểu thức điều kiện** trả về một tuple, rồi **unpack** ngay vào hai biến.

## 7. Code nộp LeetCode

In [13]:
class Solution:
    def decrypt(self, code: list[int], k: int) -> list[int]:
        n = len(code)
        res = [0] * n
        if k == 0:
            return res
        start, end = (1, k) if k > 0 else (n + k, n - 1)
        window_sum = sum(code[t % n] for t in range(start, end + 1))
        for i in range(n):
            res[i] = window_sum
            window_sum += code[(end + 1) % n] - code[start % n]
            start += 1
            end += 1
        return res

sol = Solution()
assert sol.decrypt([5, 7, 1, 4], 3) == [12, 10, 16, 13]
assert sol.decrypt([1, 2, 3, 4], 0) == [0, 0, 0, 0]
assert sol.decrypt([2, 4, 9, 3], -2) == [12, 5, 6, 13]
print("3 ví dụ của đề PASS")

3 ví dụ của đề PASS


## 8. Các lời giải khác

### 8.1. Nhân đôi mảng để khỏi dùng `%`

`doubled = code + code` dài `2n`. Mọi cửa sổ vòng của mảng gốc trở thành một **đoạn liên tiếp bình thường** trong `doubled`.

- `k > 0`: cửa sổ của `i` là `doubled[i+1 : i+k+1]`.
- `k < 0`: lấy `doubled[n + i + k : n + i]` (dịch sang nửa sau để không bị chỉ số âm).

In [14]:
def decrypt_doubled(code: list[int], k: int) -> list[int]:
    n = len(code)
    if k == 0:
        return [0] * n
    doubled = code + code
    res = []
    for i in range(n):
        if k > 0:
            res.append(sum(doubled[i + 1:i + k + 1]))
        else:
            res.append(sum(doubled[n + i + k:n + i]))
    return res

print("doubled của [2,4,9,3]:", [2, 4, 9, 3] * 2)
print(decrypt_doubled([5, 7, 1, 4], 3))
print(decrypt_doubled([2, 4, 9, 3], -2))

doubled của [2,4,9,3]: [2, 4, 9, 3, 2, 4, 9, 3]
[12, 10, 16, 13]
[12, 5, 6, 13]


Cách này vẫn `O(n · |k|)` vì mỗi `sum(slice)` cộng `|k|` số, nhưng code rất dễ đọc. Kết hợp với tổng tiền tố sẽ thành `O(n)`.

### 8.2. Tổng tiền tố (prefix sum) trên mảng nhân đôi

`prefix[t]` = tổng `doubled[0..t-1]`. Khi đó tổng đoạn `doubled[a:b]` = `prefix[b] - prefix[a]`, tính trong `O(1)`.

In [15]:
arr = [2, 4, 9, 3]
prefix = [0]
for x in arr:
    prefix.append(prefix[-1] + x)
print("arr    =", arr)
print("prefix =", prefix)
print("tổng arr[1:3] =", prefix[3] - prefix[1], "| kiểm tra:", sum(arr[1:3]))

arr    = [2, 4, 9, 3]
prefix = [0, 2, 6, 15, 18]
tổng arr[1:3] = 13 | kiểm tra: 13


In [16]:
from itertools import accumulate

def decrypt_prefix(code: list[int], k: int) -> list[int]:
    n = len(code)
    if k == 0:
        return [0] * n
    prefix = [0] + list(accumulate(code + code))
    res = []
    for i in range(n):
        a, b = (i + 1, i + k + 1) if k > 0 else (n + i + k, n + i)
        res.append(prefix[b] - prefix[a])
    return res

print(list(accumulate([2, 4, 9, 3])), "<- accumulate cộng dồn")
print(decrypt_prefix([5, 7, 1, 4], 3))
print(decrypt_prefix([2, 4, 9, 3], -2))

[2, 6, 15, 18] <- accumulate cộng dồn
[12, 10, 16, 13]
[12, 5, 6, 13]


`itertools.accumulate(xs)` trả về các tổng cộng dồn `x0, x0+x1, x0+x1+x2, ...`. Thêm `[0]` ở đầu để `prefix[0] = 0`.

### 8.3. Bảng so sánh các cách

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ | Nhận xét |
|---|---|---|---|---|
| Brute force `% n` | cộng từng phần tử | `O(n · |k|)` | `O(1)` (ngoài res) | dễ nhất, đủ pass vì `n <= 100` |
| Nhân đôi + slice | đoạn liên tiếp | `O(n · |k|)` | `O(n)` | code dễ đọc nhất |
| Sliding window | thêm mới, bỏ cũ | `O(n)` | `O(1)` (ngoài res) | **tối ưu, nên dùng** |
| Prefix sum nhân đôi | hiệu hai tiền tố | `O(n)` | `O(n)` | tổng quát cho nhiều truy vấn |

## 9. Vì sao sliding window đúng?

**Invariant:** ở đầu vòng `i`, `window_sum` bằng tổng `code[t % n]` với `t` chạy từ `start` đến `end`, và đoạn này chính là cửa sổ đề yêu cầu cho `i`.

- `i = 0`: `start`, `end` được chọn đúng theo bảng mục 4 và `window_sum` được tính trực tiếp.
- Từ `i` sang `i + 1`: cửa sổ đúng của `i + 1` là cửa sổ của `i` dịch phải 1. Bỏ phần tử ở `start`, thêm phần tử ở `end + 1`, rồi tăng cả hai → invariant giữ.

Vì `|k| <= n - 1`, cửa sổ dài `|k| < n` nên không bao giờ "đè lên chính nó" và không chứa phần tử `i` — khớp đúng với định nghĩa "next k" / "previous -k".

**Đồng thời:** ta chỉ **đọc** `code`, ghi vào `res` riêng, nên mọi tổng đều dựa trên mảng gốc.

## 10. Độ phức tạp

Gọi `n = len(code)`.

- **Sliding window:** tính tổng cửa sổ đầu `O(|k|)`, vòng trượt `O(n)` → **Time `O(n)`** (vì `|k| < n`). **Space `O(1)`** ngoài mảng kết quả `res` (bắt buộc phải trả).
- **Brute force:** `O(n · |k|)`, xấu nhất `O(n²)`.
- **Prefix sum:** `O(n)` thời gian, `O(n)` bộ nhớ cho `prefix` dài `2n + 1`.

## 11. Bộ test

1. 3 ví dụ của đề.
2. Biên: `n = 1` (khi đó `k` chỉ có thể là `0`).
3. `k = n - 1` và `k = -(n - 1)`: cửa sổ là "mọi phần tử trừ chính nó".
4. `k = 1` và `k = -1`: chỉ lấy hàng xóm.
5. Đối chiếu 4 cách trên input ngẫu nhiên với mọi `k` hợp lệ.
6. Kiểm tra `code` không bị sửa.

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<24} PASS -> {actual}")

f = decrypt_window
check("đề 1", f([5, 7, 1, 4], 3), [12, 10, 16, 13])
check("đề 2", f([1, 2, 3, 4], 0), [0, 0, 0, 0])
check("đề 3", f([2, 4, 9, 3], -2), [12, 5, 6, 13])
check("n = 1", f([42], 0), [0])
check("k = n-1", f([1, 2, 3], 2), [5, 4, 3])
check("k = -(n-1)", f([1, 2, 3], -2), [5, 4, 3])
check("k = 1 (hàng xóm phải)", f([1, 2, 3, 4], 1), [2, 3, 4, 1])
check("k = -1 (hàng xóm trái)", f([1, 2, 3, 4], -1), [4, 1, 2, 3])

original = [5, 7, 1, 4]
copy = original[:]
f(original, 3)
check("không sửa code", original, copy)

đề 1                     PASS -> [12, 10, 16, 13]
đề 2                     PASS -> [0, 0, 0, 0]
đề 3                     PASS -> [12, 5, 6, 13]
n = 1                    PASS -> [0]
k = n-1                  PASS -> [5, 4, 3]
k = -(n-1)               PASS -> [5, 4, 3]
k = 1 (hàng xóm phải)    PASS -> [2, 3, 4, 1]
k = -1 (hàng xóm trái)   PASS -> [4, 1, 2, 3]
không sửa code           PASS -> [5, 7, 1, 4]


In [18]:
import random

rng = random.Random(1652)
for _ in range(400):
    n = rng.randint(1, 12)
    code = [rng.randint(1, 100) for _ in range(n)]
    for k in range(-(n - 1), n):
        ref = decrypt_bruteforce(code, k)
        assert decrypt_window(code, k) == ref, (code, k)
        assert decrypt_doubled(code, k) == ref, (code, k)
        assert decrypt_prefix(code, k) == ref, (code, k)
        assert Solution().decrypt(code, k) == ref, (code, k)
print("4 cách + Solution khớp nhau trên 400 mảng ngẫu nhiên × mọi k hợp lệ")

4 cách + Solution khớp nhau trên 400 mảng ngẫu nhiên × mọi k hợp lệ


`original[:]` tạo **bản sao** của list (slicing toàn bộ). Nếu viết `copy = original`, hai tên trỏ cùng một list và phép so sánh sẽ vô nghĩa.

## 12. Lỗi người mới hay gặp

### Lỗi 1: Sửa trực tiếp `code` (không "đồng thời")

Khi `code[0]` đã bị thay, tổng của các `i` sau dùng **giá trị mới** → sai.

In [19]:
def wrong_in_place(code, k):
    n = len(code)
    for i in range(n):
        code[i] = sum(code[(i + j) % n] for j in range(1, k + 1))
    return code

print("sai :", wrong_in_place([5, 7, 1, 4], 3))
print("đúng:", decrypt_window([5, 7, 1, 4], 3))

sai : [12, 17, 33, 62]
đúng: [12, 10, 16, 13]


### Lỗi 2: `k < 0` mà vẫn cộng phía sau

Quên đổi chiều khi `k` âm: `range(1, k + 1)` với `k = -2` là **rỗng**, tổng ra `0`.

In [20]:
def wrong_negative(code, k):
    n = len(code)
    return [sum(code[(i + j) % n] for j in range(1, k + 1)) for i in range(n)]

print("range(1, -1) =", list(range(1, -1)))
print("sai :", wrong_negative([2, 4, 9, 3], -2))
print("đúng:", decrypt_window([2, 4, 9, 3], -2))

range(1, -1) = []
sai : [0, 0, 0, 0]
đúng: [12, 5, 6, 13]


### Lỗi 3: Quên `% n` → `IndexError` hoặc lấy nhầm

In [21]:
def wrong_no_mod(code, k):
    n = len(code)
    return [sum(code[i + j] for j in range(1, k + 1)) for i in range(n)]

try:
    wrong_no_mod([5, 7, 1, 4], 3)
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


### Lỗi 4: Cửa sổ âm bắt đầu sai chỗ

Với `k < 0`, cửa sổ của `i = 0` là `[n + k, n - 1]`, **không phải** `[k, -1]` hay `[n - 1 + k, n - 2]`. Lệch 1 vị trí là mọi kết quả đều sai.

In [22]:
def wrong_start(code, k):
    n = len(code)
    start, end = (1, k) if k > 0 else (n + k - 1, n - 2)   # lệch 1
    s = sum(code[t % n] for t in range(start, end + 1))
    res = []
    for i in range(n):
        res.append(s)
        s += code[(end + 1) % n] - code[start % n]
        start += 1
        end += 1
    return res

print("sai :", wrong_start([2, 4, 9, 3], -2))
print("đúng:", decrypt_window([2, 4, 9, 3], -2))

sai : [13, 12, 5, 6]
đúng: [12, 5, 6, 13]


### Lỗi 5: Quên `k == 0`

Với sliding window, nếu không xử lý riêng thì `k = 0` rơi vào nhánh `else`: `start = n, end = n - 1` (cửa sổ rỗng). Tình cờ code vẫn ra `0` vì mỗi bước "thêm" và "bỏ" cùng một phần tử — nhưng đó là may mắn, không phải thiết kế. Với cách prefix sum cũng vậy: đoạn rỗng `[n+i, n+i)` — ra 0 nhưng người đọc phải suy nghĩ mới hiểu. Hãy luôn trả `[0] * n` sớm cho rõ ràng.

In [23]:
n, k = 4, 0
start, end = (1, k) if k > 0 else (n + k, n - 1)
print("Nếu quên k == 0: cửa sổ đầu =", list(range(start, end + 1)), "-> cửa sổ rỗng; ra 0 chỉ là tình cờ")

Nếu quên k == 0: cửa sổ đầu = [] -> cửa sổ rỗng; ra 0 chỉ là tình cờ


## 13. Mẹo nhận dạng pattern

- Đề nói "circular" / "next of last is first" → nghĩ ngay **`% n`** hoặc **nhân đôi mảng**.
- "Tổng của k phần tử liên tiếp" cho **mọi vị trí** → **sliding window** (thêm mới, bỏ cũ) hoặc **prefix sum**.
- "Replace simultaneously" → **ghi ra mảng mới**.
- Có hai chiều (`k > 0`, `k < 0`) → quy về cùng một cửa sổ bằng cách chọn `start`, `end` phù hợp.

Bài liên quan: 643 Maximum Average Subarray I, 503 Next Greater Element II (mảng vòng), 918 Maximum Sum Circular Subarray, 1423 Maximum Points You Can Obtain from Cards.

## 14. Biến thể

### Biến thể 1: Trung bình thay vì tổng

In [24]:
def decrypt_average(code, k):
    if k == 0:
        return [0.0] * len(code)
    return [s / abs(k) for s in decrypt_window(code, k)]

print(decrypt_average([5, 7, 1, 4], 3))

[4.0, 3.3333333333333335, 5.333333333333333, 4.333333333333333]


### Biến thể 2: Tổng cửa sổ hai phía (k bên trái + k bên phải, không gồm chính nó)

In [25]:
def decrypt_both_sides(code, k):
    left = decrypt_window(code, -k)
    right = decrypt_window(code, k)
    return [a + b for a, b in zip(left, right)]

print(decrypt_both_sides([1, 2, 3, 4, 5], 1))   # mỗi số = hàng xóm trái + phải

[7, 4, 6, 8, 5]


### Biến thể 3: Max cửa sổ vòng dài `w` (dùng lại kỹ thuật trượt)

In [26]:
def max_circular_window(code, w):
    n = len(code)
    s = sum(code[:w])
    best = s
    for i in range(1, n):
        s += code[(i + w - 1) % n] - code[i - 1]
        best = max(best, s)
    return best

print(max_circular_window([5, 7, 1, 4], 2))   # cửa sổ 5+7 = 12 (cặp vòng 4+5 = 9)

12


## 15. Bài tập tự luyện

Các cell TODO chỉ định nghĩa hàm, không được gọi khi chạy notebook.

### Bài 1 — Tính tay

`code = [10, 5, 7, 7, 3, 2, 10, 3, 6, 9, 1, 6]`, `k = -4`. Tính `res[0]` và `res[1]` bằng tay. (Gợi ý: cửa sổ của `i = 0` là chỉ số `8..11`.)

### Bài 2 — Chỉ số vòng

In [27]:
def exercise_neighbors(n: int, i: int, k: int) -> list[int]:
    # TODO: trả về list chỉ số cửa sổ của i (k > 0: i+1..i+k; k < 0: i+k..i-1), đã % n
    pass

# assert exercise_neighbors(4, 3, 2) == [0, 1]
# assert exercise_neighbors(4, 0, -2) == [2, 3]

### Bài 3 — Viết lại brute force

In [28]:
def exercise_bruteforce(code: list[int], k: int) -> list[int]:
    # TODO: tạo res mới, cộng |k| phần tử theo chiều của k
    pass

# assert exercise_bruteforce([5, 7, 1, 4], 3) == [12, 10, 16, 13]
# assert exercise_bruteforce([2, 4, 9, 3], -2) == [12, 5, 6, 13]

### Bài 4 — Viết lại sliding window

In [29]:
def exercise_window(code: list[int], k: int) -> list[int]:
    # TODO 1: k == 0
    # TODO 2: start, end theo dấu k
    # TODO 3: tổng cửa sổ đầu, rồi trượt n lần
    pass

# assert exercise_window([5, 7, 1, 4], 3) == [12, 10, 16, 13]
# assert exercise_window([2, 4, 9, 3], -2) == [12, 5, 6, 13]

### Bài 5 — Biến thể: Maximum Average Subarray I (LeetCode 643, không vòng)

Trả về trung bình lớn nhất của đoạn liên tiếp dài `w` (mảng thường, không vòng).

In [30]:
def exercise_max_average(nums: list[int], w: int) -> float:
    # TODO: sliding window trên mảng thường
    pass

# assert exercise_max_average([1, 12, -5, -6, 50, 3], 4) == 12.75

### Bài 6 — Mini challenge: Giải mã ngược

Cho `res = decrypt(code, 1)`. Hãy khôi phục `code`. (Gợi ý: `res[i] = code[i+1]`; với `k = 1` việc này rất đơn giản — nhưng hãy viết cho `k = 1` trước, rồi thử nghĩ với `k = 2`.)

In [31]:
def exercise_undo_k1(res: list[int]) -> list[int]:
    # TODO: với k = 1, code[i] = res[i - 1]
    pass

# assert exercise_undo_k1(decrypt_window([5, 7, 1, 4, 2], 1)) == [5, 7, 1, 4, 2]

## 16. Gợi ý 3 mức

**Bài 1**
- Mức 1: `k = -4` → 4 phần tử phía trước.
- Mức 2: `res[0] = code[8] + code[9] + code[10] + code[11]`.
- Mức 3: `res[1] = res[0] - code[8] + code[0]`.

**Bài 2**
- Mức 1: phân biệt dấu `k`.
- Mức 2: `range(i + 1, i + k + 1)` hoặc `range(i + k, i)`.
- Mức 3: áp `% n` cho từng chỉ số.

**Bài 3**
- Mức 1: `res = [0] * n`.
- Mức 2: chiều phải `(i + j) % n`, chiều trái `(i - j) % n`.
- Mức 3: `j` chạy `1..abs(k)`.

**Bài 4**
- Mức 1: cửa sổ đầu `(1, k)` hoặc `(n + k, n - 1)`.
- Mức 2: `res[i] = s` **trước** khi trượt.
- Mức 3: `s += code[(end + 1) % n] - code[start % n]`, rồi tăng `start`, `end`.

**Bài 5**
- Mức 1: tổng `w` phần tử đầu.
- Mức 2: trượt: `s += nums[i] - nums[i - w]` với `i` từ `w`.
- Mức 3: trả `best / w`.

**Bài 6**
- Mức 1: `res[i]` chính là `code[(i + 1) % n]`.
- Mức 2: vậy `code[j] = res[(j - 1) % n]`.
- Mức 3: `[res[(j - 1) % n] for j in range(n)]`.

## 17. Đáp án tham khảo

**Bài 1:** `res[0] = 6 + 9 + 1 + 6 = 22`; `res[1] = 22 - code[8] + code[0] = 22 - 6 + 10 = 26`.

In [32]:
def answer_neighbors(n, i, k):
    if k > 0:
        return [t % n for t in range(i + 1, i + k + 1)]
    return [t % n for t in range(i + k, i)]

def answer_bruteforce(code, k):
    n = len(code)
    res = [0] * n
    for i in range(n):
        for j in range(1, abs(k) + 1):
            res[i] += code[(i + j) % n] if k > 0 else code[(i - j) % n]
    return res

def answer_window(code, k):
    n = len(code)
    res = [0] * n
    if k == 0:
        return res
    start, end = (1, k) if k > 0 else (n + k, n - 1)
    s = sum(code[t % n] for t in range(start, end + 1))
    for i in range(n):
        res[i] = s
        s += code[(end + 1) % n] - code[start % n]
        start += 1
        end += 1
    return res

def answer_max_average(nums, w):
    s = sum(nums[:w])
    best = s
    for i in range(w, len(nums)):
        s += nums[i] - nums[i - w]
        best = max(best, s)
    return best / w

def answer_undo_k1(res):
    n = len(res)
    return [res[(j - 1) % n] for j in range(n)]

big = [10, 5, 7, 7, 3, 2, 10, 3, 6, 9, 1, 6]
assert answer_window(big, -4)[:2] == [22, 26]   # Bài 1
assert answer_neighbors(4, 3, 2) == [0, 1]
assert answer_neighbors(4, 0, -2) == [2, 3]
assert answer_bruteforce([5, 7, 1, 4], 3) == [12, 10, 16, 13]
assert answer_bruteforce([2, 4, 9, 3], -2) == [12, 5, 6, 13]
assert answer_bruteforce([1, 2, 3, 4], 0) == [0, 0, 0, 0]
assert answer_window([5, 7, 1, 4], 3) == [12, 10, 16, 13]
assert answer_window([2, 4, 9, 3], -2) == [12, 5, 6, 13]
assert answer_max_average([1, 12, -5, -6, 50, 3], 4) == 12.75
assert answer_undo_k1(decrypt_window([5, 7, 1, 4, 2], 1)) == [5, 7, 1, 4, 2]
print("Tất cả đáp án PASS")
print("decrypt(big, -4) =", answer_window(big, -4))

Tất cả đáp án PASS
decrypt(big, -4) = [22, 26, 22, 28, 29, 22, 19, 22, 18, 21, 28, 19]


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def decrypt(self, code: list[int], k: int) -> list[int]:
        n = len(code)
        res = [0] * n                                   # 1) mảng mới (đồng thời)
        if k == 0:
            return res
        start, end = (1, k) if k > 0 else (n + k, n - 1)  # 2) cửa sổ của i = 0
        s = sum(code[t % n] for t in range(start, end + 1))
        for i in range(n):                              # 3) trượt
            res[i] = s
            s += code[(end + 1) % n] - code[start % n]
            start += 1
            end += 1
        return res
```

Câu thần chú: **"Mảng vòng → `% n`; tổng k liên tiếp mọi vị trí → cửa sổ trượt."**

Checklist trước khi submit:

- [ ] Ghi vào mảng **mới**, không sửa `code`?
- [ ] Xử lý `k == 0`?
- [ ] Với `k < 0`, cửa sổ đầu là `[n + k, n - 1]`?
- [ ] Mọi chỉ số đều qua `% n`?
- [ ] Ghi `res[i]` **trước** khi trượt cửa sổ?
- [ ] Đã thử `k = 1`, `k = -1`, `k = n - 1`, `k = -(n - 1)`?